In [47]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import TimeLimit
import imageio
import os
from matplotlib.gridspec import GridSpec

In [ ]:
class RangeKeepingEnv(gym.Env):
    def __init__(self, agent_speed=1.0, schedule=None):
        super(RangeKeepingEnv, self).__init__()

        self.agent_speed = agent_speed
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(8,), dtype=np.float32)

        self.agent_pos = np.array([15.0, 15.0])
        self.moving_point_pos = np.array([5.0, 5.0])

        self.desired_range_min = 5
        self.desired_range_max = 10

        self.schedule = schedule if schedule is not None else lambda t: np.array([5 + 0.1 * t, 5 + 0.1 * t])

        self.agent_life = 1
        self.opponent_life = 5

        self.agent_cooldown = 5
        self.opponent_cooldown = 5

        self.current_step = 0
        self.max_steps = 1000

    def reset(self, seed=None, options=None):
        self.agent_pos = np.random.uniform(-20, 20, size=(2,))
        self.moving_point_pos = np.random.uniform(-20, 20, size=(2,))
        self.agent_life = 1
        self.opponent_life = 5
        self.agent_cooldown = 10
        self.opponent_cooldown = 10
        self.current_step = 0
        return np.concatenate([self.agent_pos, self.moving_point_pos]), {}

    def step(self, action):
        self.current_step += 1

        if self.agent_cooldown > 0:
            self.agent_cooldown -= 1
        if self.opponent_cooldown > 0:
            self.opponent_cooldown -= 1

        action_magnitude = np.linalg.norm(action)

        if action_magnitude > 1.0:
            action = action / action_magnitude

        movement = action * self.agent_speed
        self.agent_pos += movement

        self.moving_point_pos = self.schedule(self.current_step)
        
        distance = np.linalg.norm(self.agent_pos - self.moving_point_pos)

        reward = -1.0
        terminated = False
        truncated = False

        # Opponent shoots if agent is within desired_range_min
        if distance < self.desired_range_min and self.opponent_cooldown == 0 and self.opponent_cooldown < 0:
            self.agent_life -= 1
            self.opponent_cooldown = 10

        # Agent shoots if opponent is within desired_range_max
        if distance <= self.desired_range_max and self.agent_cooldown == 0 and self.agent_cooldown < 0:
            self.opponent_life -= 5
            self.agent_cooldown = 10
            reward += 10.0  # Positive reward for hitting the opponent

        # Check for deaths and respawn
        if self.agent_life <= 0:
            self.agent_pos = np.random.uniform(-20, 20, size=(2,))
            self.agent_life = 1
            reward -= 100.0  # Penalty for agent death

        if self.opponent_life <= 0:
            self.moving_point_pos = np.random.uniform(-20, 20, size=(2,))
            self.opponent_life = 5
            reward += 100.0  # Bonus for defeating the opponent

        # Episode truncation
        if self.current_step >= self.max_steps:
            truncated = True

        state = np.concatenate([self.agent_pos, self.moving_point_pos, self.agent_life, self.agent_cooldown, self.opponent_life, self.opponent_cooldown])
        #internal_state = np.array([self.agent_life, self.agent_cooldown, self.opponent_life, self.opponent_cooldown])
        return state, reward, terminated, truncated, {}

    def render(self, mode='human'):
        print(f"Step: {self.current_step}, Agent Position: {self.agent_pos}, Opponent Position: {self.moving_point_pos}, "
              f"Agent Life: {self.agent_life}, Opponent Life: {self.opponent_life}, "
              f"Agent Cooldown: {self.agent_cooldown}, Opponent Cooldown: {self.opponent_cooldown}")


In [49]:
# Create a function to generate the GIF of the final 1000 timesteps
class RandomMovementSchedule:
    def __init__(self, speed_limit=1.0):
        self.speed_limit = speed_limit  # Limit speed to half of agent speed
        self.direction = np.random.uniform(-1, 1, size=2)  # Initial random direction
        self.speed = np.random.uniform(0, self.speed_limit)  # Initial random speed
        self.current_step = 0
        self.change_interval = 50  # Change direction and speed every 20 steps
        self.position = np.array([5.0, 5.0])  # Start the moving point at (5, 5)

    def __call__(self, t):
        if t % self.change_interval == 0:
            # Every 20 steps, randomly change speed and direction
            self.direction = np.random.uniform(-1, 1, size=2)
            self.direction /= np.linalg.norm(self.direction)  # Normalize direction vector
            self.speed = np.random.uniform(0, self.speed_limit)
        #check for out of bounds

        if self.position[0] > 20 or self.position[0] < -20:
            self.direction[0] = -self.direction[0]
        if self.position[1] > 20 or self.position[1] < -20:
            self.direction[1] = -self.direction[1]

        # Update the position based on the current speed and direction
        self.position += self.direction * self.speed
        self.current_step += 1

        return self.position

    def circular_schedule(t, radius=5.0, angular_speed=0.05, center=(0.0, 0.0)):
        x_center, y_center = center
        x = radius * np.cos(angular_speed * t) + x_center
        y = radius * np.sin(angular_speed * t) + y_center
        return np.array([x, y])

def create_gif_from_env(env, model,steps, gif_name='range_keeping.gif'):
    frames = []

    fig = plt.figure(figsize=(10, 6))
    gs = GridSpec(1, 2, width_ratios=[4, 1])  # 4:1 ratio for the main plot and sidebars

# Main plot area
    ax_main = fig.add_subplot(gs[0, 0])

# Sidebar area
    ax_sidebar = fig.add_subplot(gs[0, 1])  

    # Reset environment
    obs, _ = env.reset()

    for step in range(steps):
        action, _ = model.predict(obs)  # Use the trained model to predict actions
        obs, reward, terminated, truncated, info = env.step(action)
        ax_main.clear()
        ax_sidebar.clear()

        # Plot the moving point and agent
        moving_point = env.moving_point_pos
        agent_point = env.agent_pos

        ax_main.plot(moving_point[0], moving_point[1], 'ro', label='Opponent')  # Moving point in red
        ax_main.plot(agent_point[0], agent_point[1], 'bo', label='Agent')  # Agent in blue

        # Draw a circle showing the desired range from the moving point
        circle1 = plt.Circle((moving_point[0], moving_point[1]), env.desired_range_min, color='g', fill=False, linestyle='--', label='Desired Range')
        circle2 = plt.Circle((moving_point[0], moving_point[1]), env.desired_range_max, color='g', fill=False, linestyle='--')
        ax_main.add_patch(circle1)
        ax_main.add_patch(circle2)
        ax_main.set_xlim(-20, 20)
        ax_main.set_ylim(-20, 20)
        ax_main.set_title(f'Step {step + 1}')
        ax_main.set_xlabel('X Position')
        ax_main.set_ylabel('Y Position')
        ax_main.legend()

        # Sidebar: vertical bars
        bar_labels = ['Agent health', 'Agent Cooldown', 'Opponent Health', 'Opponent Cooldown']  # Example categories
        #bar_values = [env.agent_life, env.agent_cooldown, env.opponent_life, env.opponent_cooldown]  # Example values
        #ax_sidebar.barh(bar_labels, bar_values, color=['blue', 'red', 'green', 'purple'])
        ax_sidebar.set_xlim(0, 10)  # Adjust based on the max value of your metrics
        ax_sidebar.set_title('Stats')

        plt.draw()

        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        frames.append(frame)

        if terminated or truncated:
            break

    imageio.mimsave(gif_name, frames, fps=20)
    print(f"GIF saved as {gif_name}")

# Training and generating the final GIF
def train_agent_with_random_schedule():
    # Instantiate the random movement schedule
    random_schedule = RandomMovementSchedule(speed_limit=1.0)

    # Instantiate the environment with the random schedule
    env = RangeKeepingEnv(agent_speed=1.0, schedule=random_schedule)

    # Initialize the PPO model with MlpPolicy
    model = PPO('MlpPolicy', env, verbose=1)

    # Train the model for 10,000 steps (can adjust this as needed)
    model.learn(total_timesteps=1000)

    # Create a GIF of the trained agent's behavior during the final 1000 timesteps
    create_gif_from_env(env, model, gif_name="range_keeping_final.gif", steps=100)

# Run the training and create the GIF
if __name__ == "__main__":
    train_agent_with_random_schedule()


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


ValueError: could not broadcast input array from shape (4,) into shape (8,)